In [1]:
import torch
import torch.nn as nn

class ChromatinCNN(nn.Module):
    """
    1D CNN for binary classification of DNA sequences.
    Predicts whether a 200bp sequence is in an open or closed chromatin region.

    Input:  (batch_size, 4, seq_len)  -- 4 one-hot channels, seq_len positions
    Output: (batch_size, 2)           -- logits for [open, closed]
    """

    def __init__(self, seq_len: int = 200, num_filters_1: int = 32,
                 num_filters_2: int = 64, kernel_1: int = 12, kernel_2: int = 8):
        super().__init__()

        # Block 1: detect individual motifs (12bp window ≈ typical TF binding site)
        self.conv1 = nn.Sequential(
            nn.Conv1d(in_channels=4, out_channels=num_filters_1,
                      kernel_size=kernel_1, padding=kernel_1 // 2),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=4)
        )

        # Block 2: detect combinations of motifs
        self.conv2 = nn.Sequential(
            nn.Conv1d(in_channels=num_filters_1, out_channels=num_filters_2,
                      kernel_size=kernel_2, padding=kernel_2 // 2),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=4)
        )

        # Compute flattened size dynamically — avoids off-by-one errors
        # (padding='same' simplifies this, but computing explicitly builds intuition)
        dummy = torch.zeros(1, 4, seq_len)
        conv_out = self.conv2(self.conv1(dummy))
        flat_size = conv_out.numel()   # total elements after conv2, before FC

        # Output: binary classification
        self.classifier = nn.Linear(flat_size, 2)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.conv1(x)    # shape: (batch, 32, seq_len//4)
        x = self.conv2(x)    # shape: (batch, 64, seq_len//16)
        x = x.flatten(1)     # shape: (batch, 64 * seq_len//16)
        x = self.classifier(x)  # shape: (batch, 2)
        return x

In [2]:
import torch.optim as optim
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

def train_one_epoch(model, loader, optimizer, criterion, device):
    """Run one epoch of training. Returns mean loss."""
    model.train()  # sets dropout/batchnorm to training mode (important habit)
    total_loss = 0.0

    for sequences, labels in loader:
        sequences = sequences.to(device)  # (batch, 4, seq_len)
        labels    = labels.to(device)     # (batch,)  -- integer class indices

        # --- The 4-step training loop ---
        optimizer.zero_grad()              # 1. clear gradients from last step
        logits = model(sequences)          # 2. forward pass
        loss = criterion(logits, labels)   # 3. compute loss
        loss.backward()                    # 4a. backpropagate
        optimizer.step()                   # 4b. update weights

        total_loss += loss.item()

    return total_loss / len(loader)


def evaluate(model, loader, criterion, device):
    """Evaluate on a held-out set. Returns (mean_loss, accuracy)."""
    model.eval()  # disables dropout, fixes batchnorm running stats
    total_loss, correct, total = 0.0, 0, 0

    with torch.no_grad():  # don't track gradients -- saves memory + compute
        for sequences, labels in loader:
            sequences, labels = sequences.to(device), labels.to(device)
            logits = model(sequences)
            loss = criterion(logits, labels)

            total_loss += loss.item()
            preds = logits.argmax(dim=1)   # highest logit = predicted class
            correct += (preds == labels).sum().item()
            total += len(labels)

    return total_loss / len(loader), correct / total


def train(model, train_loader, val_loader, n_epochs=20, lr=1e-3):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)

    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    history = {'train_loss': [], 'val_loss': [], 'val_acc': []}

    for epoch in range(n_epochs):
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
        val_loss, val_acc = evaluate(model, val_loader, criterion, device)

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)

        print(f"Epoch {epoch+1:2d}/{n_epochs} | "
              f"train_loss: {train_loss:.4f} | "
              f"val_loss: {val_loss:.4f} | "
              f"val_acc: {val_acc:.3f}")

    return history


def plot_training(history):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

    ax1.plot(history['train_loss'], label='train')
    ax1.plot(history['val_loss'],   label='val')
    ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
    ax1.set_title('Loss curves'); ax1.legend()

    ax2.plot(history['val_acc'], color='green')
    ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy')
    ax2.set_title('Validation accuracy')

    plt.tight_layout()
    plt.savefig('training_curves.png', dpi=150)
    plt.show()

In [3]:
train_dataset = ChromatinDataset(positive_seqs, negative_seqs, split='train')
val_dataset   = ChromatinDataset(positive_seqs, negative_seqs, split='val')

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_dataset,   batch_size=64, shuffle=False, num_workers=2)

model = ChromatinCNN(seq_len=200)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
# Expect roughly 80,000-150,000 parameters -- small by modern standards

history = train(model, train_loader, val_loader, n_epochs=20)
plot_training(history)

NameError: name 'ChromatinDataset' is not defined